In [5]:
# ✅ Phase 5 – Step 1: Load Normalized Test Data, Classifier & Regressor Models

import pandas as pd
import numpy as np
import joblib
import os

# Define base paths
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
DATA_FILE = os.path.join(BASE_DIR, 'data', 'clean_test_FD001.csv')
SCALER_PATH = os.path.join(BASE_DIR, 'models', 'scaler_fd001.pkl')

# Classifier models (directly in models/)
CLF_PATHS = {
    "RandomForest": os.path.join(BASE_DIR, 'models', 'rf_classifier_fd001.pkl'),
    "SVM": os.path.join(BASE_DIR, 'models', 'svm_classifier_fd001.pkl'),
    "Logistic": os.path.join(BASE_DIR, 'models', 'logreg_classifier_fd001.pkl'),
    "XGBoost": os.path.join(BASE_DIR, 'models', 'xgb_classifier_fd001.pkl'),
}

# Regressor models (inside models/phase4/)
REG_PATHS = {
    "RandomForest": os.path.join(BASE_DIR, 'models', 'phase4', 'rf_regressor_fd001.pkl'),
    "Ridge": os.path.join(BASE_DIR, 'models', 'phase4', 'ridge_regressor_fd001.pkl'),
    "SVR": os.path.join(BASE_DIR, 'models', 'phase4', 'svr_regressor_fd001.pkl'),
    "XGBoost": os.path.join(BASE_DIR, 'models', 'phase4', 'xgb_regressor_fd001.pkl'),
}

# Load test dataset
df_test = pd.read_csv(DATA_FILE)
print("✅ Loaded test data:", df_test.shape)

# Normalize using saved scaler
scaler = joblib.load(SCALER_PATH)
sensor_cols = [col for col in df_test.columns if col.startswith("sensor_")]
df_test[sensor_cols] = scaler.transform(df_test[sensor_cols])
print("✅ Applied MinMaxScaler normalization")

# Load classifier and regressor models
clf_models = {name: joblib.load(path) for name, path in CLF_PATHS.items()}
reg_models = {name: joblib.load(path) for name, path in REG_PATHS.items()}
print("✅ All models loaded successfully")


✅ Loaded test data: (13096, 20)
✅ Applied MinMaxScaler normalization
✅ All models loaded successfully


In [6]:
# ✅ Phase 5 – Step 2: Compute Risk Scores for Each Model Combination

# Container to store risk scores
risk_scores = {}

# We will compute: Risk Score = failure_prob × time_left
# For classifier → prob of Stage 4 (class index 4)
# For regressor  → predicted cycles to next stage

for clf_name, clf in clf_models.items():
    # Predict probabilities for all 5 stages
    y_proba = clf.predict_proba(df_test[sensor_cols])
    failure_probs = y_proba[:, 4]  # Probability of Stage 4 (Failure)

    for reg_name, reg in reg_models.items():
        time_left = reg.predict(df_test[sensor_cols])
        raw_risk = failure_probs * time_left
        urgent_risk = failure_probs / (time_left + 1e-6)  # urgency-based

        # Save in dataframe
        col_raw = f"risk_{clf_name}_{reg_name}_raw"
        col_urgent = f"risk_{clf_name}_{reg_name}_urgent"

        df_test[col_raw] = raw_risk
        df_test[col_urgent] = urgent_risk

        # Store for future normalization
        risk_scores[(clf_name, reg_name)] = {
            'raw': raw_risk,
            'urgent': urgent_risk
        }

print("✅ Computed raw and urgency-based risk scores for all model combinations")


✅ Computed raw and urgency-based risk scores for all model combinations


In [8]:
# ✅ Phase 5 – Step 3: Normalize Risk Scores + Alerts + Save Plots

import matplotlib.pyplot as plt
import os

# Directory to save risk plots
risk_fig_dir = os.path.join(BASE_DIR, 'figures', 'phase5')
os.makedirs(risk_fig_dir, exist_ok=True)

for (clf_name, reg_name), scores in risk_scores.items():
    raw = scores['raw']
    urgent = scores['urgent']

    # --- Min-Max Normalization ---
    raw_min, raw_max = raw.min(), raw.max()
    norm_raw = (raw - raw_min) / (raw_max - raw_min + 1e-6)

    col_norm_raw = f"norm_risk_{clf_name}_{reg_name}"
    df_test[col_norm_raw] = norm_raw

    # --- Alerts: High Risk Threshold ---
    df_test[f"alert_{clf_name}_{reg_name}"] = (norm_raw > 0.7).astype(int)

    # --- Plot risk trend over time for each engine ---
    for unit_id in df_test['unit'].unique():
        df_engine = df_test[df_test['unit'] == unit_id]

        plt.figure(figsize=(8, 4))
        plt.plot(df_engine['time'], df_engine[col_norm_raw], label='Normalized Risk Score', color='crimson')
        plt.axhline(0.7, linestyle='--', color='black', label='Threshold (0.7)')
        plt.title(f"Risk Score – Engine {unit_id} ({clf_name}+{reg_name})")
        plt.xlabel("Time (Cycles)")
        plt.ylabel("Risk Score")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()

        fig_path = os.path.join(
            risk_fig_dir,
            f"risk_trend_{clf_name}_{reg_name}_engine_{unit_id:03d}_fd001.png"
        )
        plt.savefig(fig_path)
        plt.close()

print(f"✅ Risk normalization, alert tagging, and risk trend plots saved to: {risk_fig_dir}")


✅ Risk normalization, alert tagging, and risk trend plots saved to: c:\Users\pandr\hybrid-predictive-maintenance\figures\phase5
